In [7]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score


In [8]:
df = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/train.csv')
df2 = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/test.csv')

df['is_test'] = 0
df2['is_test'] = 1

full = pd.concat([df, df2])
full = full.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)
full.shape

(227656, 11)

In [9]:
full['is_first_txn'] = (full['hours_since_prev_txn'] == 999).astype(int)
full['hours_since_prev_txn'] = full['hours_since_prev_txn'].replace(999, np.nan)


In [11]:
full['running_total'] = full.groupby('user_id')['amount'].cumsum()

full['past_total'] = full['running_total'] - full['amount'] 
full['past_count'] = full.groupby('user_id').cumcount()
full['past_avg_amount'] = full['past_total'] / full['past_count'] 
full['amount_ratio'] = full['amount'] / full['past_avg_amount']

filtered_user = full[full['user_id'] == 1818]
filtered_user[['amount', 'past_avg_amount', 'amount_ratio']]

full = full.drop(columns=['running_total', 'past_total', 'past_count'])

In [12]:
filtered_user[['amount', 'past_avg_amount', 'amount_ratio']]

,amount,past_avg_amount,amount_ratio
82574,17.788061,NaN,NaN
82575,33.010191,17.788061,1.855750
82576,37.474351,25.399126,1.475419
82577,15.018893,29.424201,0.510427
82578,10.954236,25.822874,0.424207
82579,5.410091,22.849147,0.236774
82580,14.351851,19.942637,0.719657
82581,10.819577,19.143953,0.565169
82582,5.095317,18.103406,0.281456
82583,16.608058,16.658063,0.996998


In [ ]:
full['device_seen_count'] = full.groupby(['user_id', 'device_id']).cumcount()
full['is_new_device']